In [8]:
!ls /kaggle/input

competitions


In [9]:
!ls /kaggle/input

competitions


In [10]:
!ls /kaggle/input/competitions

vinbigdata-chest-xray-abnormalities-detection


In [11]:
import pandas as pd

annotations = pd.read_csv(
    "/kaggle/input/competitions/vinbigdata-chest-xray-abnormalities-detection/train.csv"
)

print("Total annotations:", len(annotations))
print("Total unique images:", annotations["image_id"].nunique())
print("Total labels:", annotations["class_name"].nunique())

Total annotations: 67914
Total unique images: 15000
Total labels: 15


In [12]:
!pip install pydicom opencv-python tqdm albumentations

In [13]:
import pydicom

dicom_folder = "/kaggle/input/competitions/vinbigdata-chest-xray-abnormalities-detection/train"

sample_id = annotations.iloc[0]["image_id"]
dicom_path = f"{dicom_folder}/{sample_id}.dicom"

dicom = pydicom.dcmread(dicom_path)
image = dicom.pixel_array

print("Image shape:", image.shape)

Image shape: (2580, 2332)


In [14]:
import cv2
import numpy as np
import pydicom

def preprocess_image(dicom_path):

    dicom = pydicom.dcmread(dicom_path)
    image = dicom.pixel_array.astype(np.float32)

    # Step 1: Min-Max Normalization
    image = (image - image.min()) / (image.max() - image.min())
    image = (image * 255).astype(np.uint8)

    # Step 2: CLAHE (Contrast Enhancement)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    image = clahe.apply(image)

    # Step 3: Light Gaussian Denoising
    image = cv2.GaussianBlur(image, (3,3), 0)

    # Step 4: Resize to 512x512
    image = cv2.resize(image, (512, 512))

    return image

In [16]:
import pydicom
import cv2
import numpy as np

def dicom_to_png(dicom_path):
    dicom = pydicom.dcmread(dicom_path)
    image = dicom.pixel_array
    return image

In [ ]:
def preprocess_image(dicom_path):

    #  DICOM → image
    image = dicom_to_png(dicom_path)

    #  Convert to grayscale (already grayscale but keep step as required)
    if len(image.shape) == 3:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    #  Resize
    image = cv2.resize(image, (512, 512))

    #  Min-Max Normalization
    image = (image - image.min()) / (image.max() - image.min())
    image = (image * 255).astype(np.uint8)

    #  CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    image = clahe.apply(image)

    #  Light Gaussian Denoising
    image = cv2.GaussianBlur(image, (3,3), 0)

    #  Crop Borders
    image = image[5:-5, 5:-5]

    # Padding to uniform shape
    image = cv2.copyMakeBorder(
        image, 5,5,5,5,
        cv2.BORDER_CONSTANT,
        value=0
    )

    return image

In [ ]:
import numpy as np
import pandas as pd

images_per_label = 444  
selected_images = set()

for label in annotations["class_name"].unique():
    
    label_df = annotations[annotations["class_name"] == label]
    
    unique_imgs = label_df["image_id"].unique()
    
    take = min(images_per_label, len(unique_imgs))
    
    chosen = np.random.choice(unique_imgs, take, replace=False)
    
    selected_images.update(chosen)

selected_images = list(selected_images)

print("Total selected images:", len(selected_images))

In [23]:
import numpy as np

images_per_label = 333  # strict per class
selected_images = []

for label in annotations["class_name"].unique():
    
    label_df = annotations[annotations["class_name"] == label]
    unique_imgs = label_df["image_id"].unique()
    
    take = min(images_per_label, len(unique_imgs))
    
    chosen = np.random.choice(unique_imgs, take, replace=False)
    
    # DO NOT remove duplicates (strict per-label logic)
    selected_images.extend(chosen)

print("Total entries ", len(selected_images))

Total entries  4611


In [24]:
from tqdm import tqdm
import os

dicom_folder = "/kaggle/input/competitions/vinbigdata-chest-xray-abnormalities-detection/train"

os.makedirs("processed/images", exist_ok=True)

for img_id in tqdm(selected_images):
    
    dicom_path = f"{dicom_folder}/{img_id}.dicom"
    
    image = preprocess_image(dicom_path)
    
    cv2.imwrite(f"processed/images/{img_id}.png", image)

100%|██████████| 4611/4611 [1:13:58<00:00,  1.04it/s]


In [25]:
os.makedirs("processed/labels", exist_ok=True)

In [26]:
def convert_to_yolo(image_id, df):
    
    image_df = df[df["image_id"] == image_id]
    yolo_lines = []
    
    original_height = 2580
    original_width = 2332
    
    for _, row in image_df.iterrows():
        
        # Skip if no bounding box
        if row["class_id"] == 14:  # 14 = No finding
            continue
        
        # Scale bounding box to 512x512
        x_min = row["x_min"] * (512 / original_width)
        x_max = row["x_max"] * (512 / original_width)
        y_min = row["y_min"] * (512 / original_height)
        y_max = row["y_max"] * (512 / original_height)
        
        # Convert to YOLO format
        x_center = ((x_min + x_max) / 2) / 512
        y_center = ((y_min + y_max) / 2) / 512
        width = (x_max - x_min) / 512
        height = (y_max - y_min) / 512
        
        yolo_lines.append(
            f"{int(row['class_id'])} {x_center} {y_center} {width} {height}"
        )
    
    return yolo_lines

In [27]:
from tqdm import tqdm

for img_id in tqdm(selected_images):
    
    yolo_labels = convert_to_yolo(img_id, annotations)
    
    with open(f"processed/labels/{img_id}.txt", "w") as f:
        for line in yolo_labels:
            f.write(line + "\n")

100%|██████████| 4611/4611 [00:26<00:00, 171.31it/s]


In [28]:
from sklearn.model_selection import train_test_split

train_ids, temp_ids = train_test_split(
    selected_images, test_size=0.3, random_state=42
)

val_ids, test_ids = train_test_split(
    temp_ids, test_size=0.5, random_state=42
)

print("Train:", len(train_ids))
print("Val:", len(val_ids))
print("Test:", len(test_ids))

Train: 3227
Val: 692
Test: 692


In [29]:
import os

base_path = "dataset"

folders = [
    "dataset/images/train",
    "dataset/images/val",
    "dataset/images/test",
    "dataset/labels/train",
    "dataset/labels/val",
    "dataset/labels/test"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

In [30]:
import shutil
from tqdm import tqdm

def move_files(image_ids, split_name):
    
    for img_id in tqdm(image_ids):
        
        src_img = f"processed/images/{img_id}.png"
        src_lbl = f"processed/labels/{img_id}.txt"
        
        dst_img = f"dataset/images/{split_name}/{img_id}.png"
        dst_lbl = f"dataset/labels/{split_name}/{img_id}.txt"
        
        if os.path.exists(src_img):
            shutil.copy(src_img, dst_img)
        
        if os.path.exists(src_lbl):
            shutil.copy(src_lbl, dst_lbl)

move_files(train_ids, "train")
move_files(val_ids, "val")
move_files(test_ids, "test")

100%|██████████| 692/692 [00:00<00:00, 3032.90it/s]


In [31]:
import os

folders = [
    "dataset_augmented/images/train",
    "dataset_augmented/images/val",
    "dataset_augmented/images/test",
    "dataset_augmented/labels/train",
    "dataset_augmented/labels/val",
    "dataset_augmented/labels/test"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

In [32]:
import shutil
from tqdm import tqdm

def copy_split(image_ids, split):
    for img_id in tqdm(image_ids):
        shutil.copy(
            f"dataset/images/{split}/{img_id}.png",
            f"dataset_augmented/images/{split}/{img_id}.png"
        )
        shutil.copy(
            f"dataset/labels/{split}/{img_id}.txt",
            f"dataset_augmented/labels/{split}/{img_id}.txt"
        )

copy_split(val_ids, "val")
copy_split(test_ids, "test")

100%|██████████| 692/692 [00:00<00:00, 3972.77it/s]


In [34]:
import albumentations as A

train_transform = A.Compose(
    [
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=10, p=0.5),
        A.RandomBrightnessContrast(p=0.3),
        A.RandomScale(scale_limit=0.1, p=0.3),
        A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.2),
    ],
    bbox_params=A.BboxParams(
        format="yolo",
        label_fields=["class_labels"]
    )
)

/tmp/ipykernel_55/3651587046.py:9: UserWarning: Argument(s) 'alpha_affine' are not valid for transform ElasticTransform
  A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.2),


In [35]:
import albumentations as A

train_transform = A.Compose(
    [
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=10, p=0.5),
        A.RandomBrightnessContrast(p=0.3),
        A.RandomScale(scale_limit=0.1, p=0.3),
        A.ElasticTransform(alpha=1, sigma=50, p=0.2),
    ],
    bbox_params=A.BboxParams(
        format="yolo",
        label_fields=["class_labels"]
    )
)

In [38]:
import albumentations as A

train_transform = A.Compose(
    [
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=10, p=0.5),
        A.RandomBrightnessContrast(p=0.3),
        A.RandomScale(scale_limit=0.1, p=0.3),
        A.ElasticTransform(alpha=1, sigma=50, p=0.2),
    ],
    bbox_params=A.BboxParams(
        format="yolo",
        label_fields=["class_labels"]
    )
)

In [39]:
import cv2
import shutil
import numpy as np
from tqdm import tqdm
import os

# Ensure augmented folders exist
os.makedirs("dataset_augmented/images/train", exist_ok=True)
os.makedirs("dataset_augmented/labels/train", exist_ok=True)

for img_id in tqdm(train_ids):

    img_path = f"dataset/images/train/{img_id}.png"
    lbl_path = f"dataset/labels/train/{img_id}.txt"

    image = cv2.imread(img_path)

    boxes = []
    class_labels = []

    # -------- LOAD + VALIDATE YOLO BOXES --------
    with open(lbl_path, "r") as f:
        for line in f.readlines():
            parts = line.strip().split()
            cls = int(parts[0])
            x, y, w, h = map(float, parts[1:])

            # Validate YOLO format strictly
            if (
                0 < x < 1 and
                0 < y < 1 and
                0 < w < 1 and
                0 < h < 1 and
                (x - w/2) >= 0 and
                (x + w/2) <= 1 and
                (y - h/2) >= 0 and
                (y + h/2) <= 1
            ):
                boxes.append([x, y, w, h])
                class_labels.append(cls)

    # -------- SAVE ORIGINAL --------
    shutil.copy(img_path, f"dataset_augmented/images/train/{img_id}.png")
    shutil.copy(lbl_path, f"dataset_augmented/labels/train/{img_id}.txt")

    # If no valid boxes → skip augmentation
    if len(boxes) == 0:
        continue

    # -------- APPLY AUGMENTATION --------
    augmented = train_transform(
        image=image,
        bboxes=boxes,
        class_labels=class_labels
    )

    aug_image = augmented["image"]
    aug_boxes = augmented["bboxes"]
    aug_labels = augmented["class_labels"]

    # If augmentation removed all boxes → skip
    if len(aug_boxes) == 0:
        continue

    aug_id = img_id + "_aug"

    # -------- SAVE AUGMENTED IMAGE --------
    cv2.imwrite(
        f"dataset_augmented/images/train/{aug_id}.png",
        aug_image
    )

    # -------- SAVE AUGMENTED LABEL --------
    with open(
        f"dataset_augmented/labels/train/{aug_id}.txt", "w"
    ) as f:
        for cls, box in zip(aug_labels, aug_boxes):
            f.write(f"{cls} {' '.join(map(str, box))}\n")

100%|██████████| 3227/3227 [01:56<00:00, 27.75it/s]
